# DescriPyTor: modelling with BASSA

BASSA is a Bayesian spike-and-slab regression. It does not return one best model. It samples many small linear models and
reports how often each descriptor is used (its **inclusion frequency**), with which sign, and which descriptor sets come up
most. Predictions average over all of them. On 15 to 40 compounds and a pool of tens of descriptors, this shows how many
different models the data support, which a single "best" equation hides.

**Once, before the first run:** the sampler needs its own Python, because `bassa-reg` pins NumPy 2 and DescriPyTor needs
NumPy < 2. Run this in a terminal (it takes a few minutes):

```bash
descripytor bassa_setup
```

This notebook then runs in your usual DescriPyTor kernel. `M3_modeler.bassa` sends each table to the sampler's Python and
brings the answer back.

**Workflow**
1. Load a CSV of features and a response, and split it into X and y.
2. Audit the pool, without reading the response.
3. Fit BASSA on all compounds: which descriptors, which sign, which models.
4. BASSA's figures: the model arrangement plot, the Markov chain, the survival plot and the convergence plots.
5. Cross-validate with the sampler rerun in every fold. This is the number to report.
6. Next steps: the explorer, the full protocol, the controls.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

try:
    import M3_modeler
except ModuleNotFoundError:                      # running from a clone without `pip install -e .`
    sys.path.insert(0, str(Path().resolve().parent))
    import M3_modeler
from M3_modeler import audit, bassa

print("DescriPyTor modules from", Path(bassa.__file__).parent)

## 1. Load the CSV and split it into X and y

The CSV has the compound names in the first column, then the descriptors, and one response column. `bassa.load_table`
finds the response by name: the first column called `output`, `y`, `ddG`, `response`, `target`, `yield`, `ee` or `er`
(any case), otherwise the last column. Pass `response="..."` to name it yourself. Text columns, columns with missing
values and compounds without a response are dropped, and each drop is printed.

The example ships with the package. It is Case Study 2 of the DescriPyTor paper: 17 ligands, ΔΔG‡ in kcal/mol, and 50
descriptors read from DFT structures (Sterimol on several axes, charges, framed dipole components). For your own data,
replace the path.

In [ ]:
csv_path = Path(bassa.__file__).parent / "bassa_explorer" / "example_cs2.csv"   # your CSV here

X, y = bassa.load_table(csv_path)          # or bassa.load_table(csv_path, response="output")
X.iloc[:5, :6]

## 2. Audit the pool first (the response is not read)

`zaudit` flags columns where one compound sits far from the others (more than 6 leave-one-out SDs).
- **degenerate:** without that compound the column has almost no spread left. Drop it before any fit. A held-out fold
  would otherwise put that compound tens of SDs away.
- **outlier:** the column has spread of its own, and the compound is unusual. Look at that structure before trusting it.

Constant columns are dropped by the sampler anyway.

In [ ]:
flags = audit.zaudit(X)
flags if len(flags) else "no column is dominated by one compound"

In [ ]:
degenerate = flags.loc[flags.kind == "degenerate", "column"].tolist()
X = X.drop(columns=degenerate)
print(f"dropped {degenerate}; {X.shape[1]} descriptors left")

## 3. Fit on all compounds

One sampler run with the set-up of the BASSA explorer: corrected core, a prior expecting 3 terms, 20,000 steps with the
first fifth dropped, seed 7. It takes a minute or two. The output is:
- **inclusion:** the share of steps whose model contains the descriptor. Above about 0.5 the data keep asking for it.
- **positive:** when the descriptor is included, the share of steps with a positive weight. Near 1 or 0 means a stable sign.
- **coef:** the mean weight when included, in response units per descriptor unit.
- **models:** the descriptor sets the sampler visited most, with their share of steps. A small top share is normal; the
  averaged model uses all of them.

These numbers are in-sample. They show what the data support, not how well the model predicts.

In [ ]:
r = bassa.fit(X, y)
print(f"mean model size: {r['size']:.1f} terms")
r["table"].head(10).round(3)

In [ ]:
r["models"].assign(share=lambda d: d.share.round(3))

## 4. BASSA's figures

`bassa.figures` runs the sampler once more with saving switched on (same set-up and seed), then BASSA's own report, and
returns the PNGs. These are the figures of the BASSA README:
- **Model arrangement** (`bassa_plot`): the models BASSA keeps, as columns, with their training R² and the descriptors each
  one uses. Only models with training R² of at least 0.5 are kept.
- **Markov chain** (`markov_chain_visualization`): which of the top 20 descriptors are in the model at each step.
- **Survival** (`survival_plot`): how long each model persists as the descriptor cut-off is raised.
- **Convergence** (`sparsity_convergence`, `log_likelihood_convergence`): the last 500 steps of the model size and of the
  log-density, with their autocorrelation. Flat traces and a fast-decaying autocorrelation mean the chain mixed.

Everything, including the interactive HTML viewer `bassa_plot_viewer/index.html`, is written to the folder printed below.
On Windows keep that folder's path short: BASSA's files fail beyond 260 characters.

In [ ]:
figs = bassa.figures(X, y, "bassa_figures")
print("written to", figs["folder"])
for name in bassa.FIGURES:
    if name in figs:
        display(Markdown(f"**{name}**"), Image(filename=str(figs[name]), width=820))

## 5. Cross-validate: the number to report

The R² values in the model arrangement plot are training values. `cross_validate` splits the compounds into 5 folds,
stratified by the response, and **reruns the sampler on every training fold**. No choice made on all compounds reaches the
held-out ones, so the Q² includes the descriptor selection. Each held-out compound also gets a 90 % interval from the
sampler's predictive draws, and about 90 % of them should fall inside.

Five sampler runs, several minutes. `deals=` repeats it with other fold assignments. The paper uses 20 deals plus
leave-one-out, through `descripytor bassa_protocol` (below).

In [ ]:
cv = bassa.cross_validate(X, y, deals=1)
print(f"5-fold Q2 {cv['q2']:.2f}, MAE {cv['mae']:.2f}, 90 % intervals hold {100 * cv['coverage']:.0f} % of compounds")

In [ ]:
p = cv["predictions"]
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.errorbar(p.y, p.pred, yerr=[p.pred - p.lo, p.hi - p.pred], fmt="o", ms=5, color="#2b6cb0", ecolor="#90cdf4", capsize=2)
lim = [min(p.y.min(), p.lo.min()) - 0.2, max(p.y.max(), p.hi.max()) + 0.2]
ax.plot(lim, lim, color="0.6", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel(f"measured {y.name}"); ax.set_ylabel(f"predicted {y.name} (held out)")
ax.set_title(f"5-fold, sampler rerun in every fold: Q² {cv['q2']:.2f}", fontsize=9)
plt.tight_layout()

## 6. Next steps

**See the models on the molecules.** `descripytor bassa_explorer` opens the BASSA model explorer. Choose *New case* and
load the same CSV, with a folder of structures if you have one. Each column is a visited model, and the selected
descriptors are drawn on a compound.

**The full validation of a kept case** (20 deals of 5-fold plus leave-one-out, the chance line on shuffled responses, a
second sampler seed, and drop-one: does one compound carry the model?):

```bash
descripytor bassa_protocol protocol my_case
descripytor bassa_protocol chance my_case
descripytor bassa_protocol drop_one my_case
```

**Before you write a claim:**
- Compare with a baseline on the same folds. `M3_modeler.validation.paired_sign_flip` tests which model is closer,
  compound by compound. Use "beats" only with p < 0.05, and "matches" otherwise.
- Equal Q² is not equivalence. `validation.equivalence` (a TOST test) usually says *underpowered* at this size, so write
  "no detectable difference".
- A descriptor family that "predicts" should beat random families of the same size: `validation.family_null`.
- Interval claims: `validation.calibration` on the protocol's json.

**Your own table:** keep the pool written down before you look at the response, and cut a wide table with
`audit.reduce_pool(X, 50)`, which never reads it.